# **DATASET & PREPROCESSING**

In [65]:
import nltk
from nltk.corpus import conll2002
from sklearn.metrics import accuracy_score, f1_score

# Download and Load the Data
nltk.download('conll2002', quiet=True)
train_sents = conll2002.iob_sents('esp.train')
test_sents = conll2002.iob_sents('esp.testa')

# Helper function to prepare data for POS Tagging (Word, Tag tuple)
def prepare_pos(sents):
    return [[(w, t) for w, t, ner in s] for s in sents]

train_pos = prepare_pos(train_sents)
test_pos = prepare_pos(test_sents)

print(f"Train Dataset Size: {len(train_pos)} sentences")
print(f"Test Dataset Size: {len(test_pos)} sentences")

Train Dataset Size: 8323 sentences
Test Dataset Size: 1915 sentences


# **Method 0 - Unigram**

In [66]:
from nltk.tag import UnigramTagger, DefaultTagger

default_tagger = DefaultTagger('NC')

uni_tagger = UnigramTagger(train_pos, backoff=default_tagger)

# Evaluate the model
y_true = [tag for sent in test_pos for word, tag in sent]
y_pred = [tag for sent in uni_tagger.tag_sents([[w for w,t in s] for s in test_pos]) for w,tag in sent]

print(f" Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f" F1 Score: {f1_score(y_true, y_pred, average='weighted'):.4f}")

 Accuracy: 0.9028
 F1 Score: 0.8972


# **METHOD 1 : HIDDEN MARKOV MODEL (HMM)**

In [67]:
from nltk.tag import HiddenMarkovModelTagger


hmm_tagger = HiddenMarkovModelTagger.train(train_pos)

#  Predict on Test Data
# We extract only words for prediction to be fair
test_words = [[w for w,t in s] for s in test_pos]
y_pred_hmm = [tag for sent in hmm_tagger.tag_sents(test_words) for w,tag in sent]

#  Print Results
print(f"HMM Accuracy: {accuracy_score(y_true, y_pred_hmm):.4f}")
print(f"HMM F1 Score: {f1_score(y_true, y_pred_hmm, average='weighted'):.4f}")

HMM Accuracy: 0.9027
HMM F1 Score: 0.9023


# **Method 2 - Perceptron**

In [5]:
from nltk.tag import PerceptronTagger

# 1. Initialize and Train
pct_tagger = PerceptronTagger(load=False)
pct_tagger.train(train_pos)

# 2. Predict
test_words = [[w for w,t in s] for s in test_pos]
y_pred_pct = [tag for sent in pct_tagger.tag_sents(test_words) for w,tag in sent]

# 3. Results
print(f"Perceptron Accuracy: {accuracy_score(y_true, y_pred_pct):.4f}")
print(f"Perceptron F1 Score: {f1_score(y_true, y_pred_pct, average='weighted'):.4f}")

Perceptron Accuracy: 0.9551
Perceptron F1 Score: 0.9544


# **Method 3 - Bigram Tagger**

In [6]:
from nltk.tag import BigramTagger

# 1. Train Bigram Tagger
# We use the previously trained 'uni_tagger' as a backoff.
# If the Bigram model doesn't know a pair, it asks the Unigram model.
print("Training Bigram Tagger with Backoff...")
bigram_tagger = BigramTagger(train_pos, backoff=uni_tagger)

# 2. Predict on Test Data
test_words = [[w for w,t in s] for s in test_pos]
y_pred_bi = [tag for sent in bigram_tagger.tag_sents(test_words) for w,tag in sent]

# 3. Evaluation
acc_bi = accuracy_score(y_true, y_pred_bi)
f1_bi = f1_score(y_true, y_pred_bi, average='weighted')

print(f"Bigram Accuracy: {acc_bi:.4f}")
print(f"Bigram F1 Score: {f1_bi:.4f}")

Training Bigram Tagger with Backoff...
Bigram Accuracy: 0.9176
Bigram F1 Score: 0.9146


# **Method 4 - CRF**

In [17]:
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.metrics import accuracy_score

# 1. Train CRF Model
def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features[':-1.word.lower()'] = word1.lower()
        features[':-1.word.istitle()'] = word1.istitle()
        features[':-1.word.isupper()'] = word1.isupper()
        features[':-1.postag'] = postag1
        features[':-1.postag[:2]'] = postag1[:2]
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features[':+1.word.lower()'] = word1.lower()
        features[':+1.word.istitle()'] = word1.istitle()
        features[':+1.word.isupper()'] = word1.isupper()
        features[':+1.postag'] = postag1
        features[':+1.postag[:2]'] = postag1[:2]
    else:
        features['EOS'] = True

    return features

def sent2labels(sent):
    return [label for word, postag, label in sent]

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

# Prepare training data for CRF
X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]
crf = sklearn_crfsuite.CRF(algorithm='lbfgs', max_iterations=100)
crf.fit(X_train, y_train)

# 2. Predict on Test Set
X_test = [[word2features(s, i) for i in range(len(s))] for s in test_sents]
y_test = [[label for w, p, label in s] for s in test_sents]
y_pred_crf = crf.predict(X_test)

# 3. Calculate F1 Score and Accuracy
labels = list(crf.classes_)
labels.remove('O')
f1 = metrics.flat_f1_score(y_test, y_pred_crf, average='weighted', labels=labels)

# Flatten the lists for accuracy calculation
y_test_flat = [label for sublist in y_test for label in sublist]
y_pred_crf_flat = [label for sublist in y_pred_crf for label in sublist]
accuracy = accuracy_score(y_test_flat, y_pred_crf_flat)

print(f"CRF Accuracy: {accuracy:.4f}")
print(f"CRF F1 Score: {f1:.4f}")

CRF Accuracy: 0.9569
CRF F1 Score: 0.7393


# **Method 5 - BERT**

In [48]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from seqeval.metrics import classification_report, f1_score, accuracy_score

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("mrm8488/bert-spanish-cased-finetuned-ner", use_fast=True)
model = AutoModelForTokenClassification.from_pretrained("mrm8488/bert-spanish-cased-finetuned-ner")
model.eval()

# Prepare plain text sentences
test_texts = [" ".join([w for w, t, n in sent]) for sent in test_sents]

# Predict word-level IOB2 labels
def predict_iob2(tokens, text):
    words = [w for w, _, _ in tokens]

    if len(words) == 0:
        return []

    encoded = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=False
    )

    word_ids = encoded.word_ids()

    with torch.no_grad():
        output = model(**{k: v for k, v in encoded.items() if k != "offset_mapping"})

    logits = output.logits[0]
    pred_ids = torch.argmax(logits, dim=-1).tolist()

    # Collect all subword predictions for each word
    word_predictions = {}
    for idx, wid in enumerate(word_ids):
        if wid is None or wid < 0 or wid >= len(words):
            continue

        if wid not in word_predictions:
            word_predictions[wid] = []

        label_id = pred_ids[idx]
        raw_label = model.config.id2label[label_id]
        word_predictions[wid].append(raw_label)

    # Select the best label for each word
    result = []
    prev_entity_type = None

    for wid in range(len(words)):
        if wid not in word_predictions or len(word_predictions[wid]) == 0:
            result.append("O")
            prev_entity_type = None
            continue

        # Take the label of the first subword
        first_label = word_predictions[wid][0]

        if first_label == "O":
            result.append("O")
            prev_entity_type = None
        else:
            # Extract entity type
            if first_label.startswith("B-") or first_label.startswith("I-"):
                entity_type = first_label[2:]
            else:
                entity_type = first_label

            # Use I- if the previous word has the same entity type
            if prev_entity_type == entity_type:
                result.append("I-" + entity_type)
            else:
                result.append("B-" + entity_type)

            prev_entity_type = entity_type

    return result

# Predict all sentences
y_pred = []
y_true = []

for tokens, text in zip(test_sents, test_texts):
    pred = predict_iob2(tokens, text)
    true = [ner for (_, _, ner) in tokens]

    # Length check
    if len(pred) != len(true):
        if len(pred) < len(true):
            pred.extend(["O"] * (len(true) - len(pred)))
        else:
            pred = pred[:len(true)]

    y_pred.append(pred)
    y_true.append(true)

# Show first few examples
print("\n=== PREDICTION EXAMPLE ===")
for i in range(min(2, len(y_pred))):
    print(f"\nSentence {i+1}:")
    words = [w for w, _, _ in test_sents[i]]
    for w, t, p in zip(words, y_true[i], y_pred[i]):
        if t != "O" or p != "O":
            print(f"  {w:20s} -> True: {t:10s} Pred: {p:10s} {'✓' if t==p else '✗'}")

# Evaluate
print("\n=== EVALUATION RESULTS ===")
print(classification_report(y_true, y_pred))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



=== PREDICTION EXAMPLES ===

Sentence 1:
  Sao                  -> True: B-LOC      Pred: B-LOC      ✓
  Paulo                -> True: I-LOC      Pred: I-LOC      ✓
  Brasil               -> True: B-LOC      Pred: B-LOC      ✓
  EFECOM               -> True: B-ORG      Pred: B-ORG      ✓

Sentence 2:

=== EVALUATION RESULTS ===
              precision    recall  f1-score   support

         LOC       0.85      0.81      0.83       985
        MISC       0.77      0.76      0.77       445
         ORG       0.82      0.80      0.81      1700
         PER       0.81      0.79      0.80      1222

   micro avg       0.82      0.80      0.81      4352
   macro avg       0.81      0.79      0.80      4352
weighted avg       0.82      0.80      0.81      4352

Accuracy: 0.9615668046029137
F1: 0.8064253288324991


# **Method 6: SpaCy (CNN-based Pipeline)**

In [47]:
import spacy
from spacy.tokens import Doc
from sklearn.metrics import accuracy_score, f1_score

# 1. Load Model
print("Loading SpaCy Model...")
nlp = spacy.load("es_core_news_sm")

# 2. Qualitative Demo (Single Sentence)
print("\n--- [Qualitative] Single Example ---")
text = "Apple está buscando comprar una startup en Madrid."
doc_demo = nlp(text)
for ent in doc_demo.ents:
    print(f"Entity: {ent.text} | Label: {ent.label_}")

# 3. Evaluation
print("\n--- [Quantitative] Evaluating on Test Set ---")

y_true = []
y_pred = []


for sent in test_sents:

    words = [token[0] for token in sent]
    labels = [token[2] for token in sent]



    doc = Doc(nlp.vocab, words=words)


    nlp.get_pipe("ner")(doc)


    preds = [t.ent_type_ if t.ent_type_ else 'O' for t in doc]

    y_true.extend(labels)
    y_pred.extend(preds)

# 4. Calculate Metrics
print(f"SpaCy Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"SpaCy F1 Score: {f1_score(y_true, y_pred, average='weighted'):.4f}")

Loading SpaCy Model...

--- [Qualitative] Single Example ---
Entity: Apple | Label: ORG
Entity: Madrid | Label: LOC

--- [Quantitative] Evaluating on Test Set ---
SpaCy Accuracy: 0.8422
SpaCy F1 Score: 0.8435


In [54]:
import pandas as pd

# Data
data = {
    "Algorithm": [
        "Unigram (Baseline)", "HMM", "Bigram (Context)", "Perceptron",
        "CRF", "BERT (Transformer)", "SpaCy (CNN)"
    ],
    "Task": [
        "POS Tagging", "POS Tagging", "POS Tagging", "POS Tagging",
        "NER", "NER", "NER"
    ],
    "Accuracy": [0.9028, 0.9027, 0.9176, 0.9551, 0.9569, 0.9616, 0.8422],
    "F1 Score": [0.8972, 0.9023, 0.9146, 0.9544, 0.7393, 0.8064, 0.8435]
}

# Create DataFrame
df = pd.DataFrame(data)

# Display floats as percentages
pd.options.display.float_format = '{:.2%}'.format

# Table 1: POS Tagging
print("=== TABLE 1: POS TAGGING RESULTS ===\n")
df_pos = df[df["Task"] == "POS Tagging"].sort_values(by="F1 Score", ascending=False)
print(df_pos[["Algorithm", "Accuracy", "F1 Score"]].to_markdown(index=False))

print("\n" + "="*60 + "\n")

# Table 2: NER
print("=== TABLE 2: NER RESULTS ===\n")
df_ner = df[df["Task"] == "NER"].sort_values(by="F1 Score", ascending=False)
print(df_ner[["Algorithm", "Accuracy", "F1 Score"]].to_markdown(index=False))


=== TABLE 1: POS TAGGING RESULTS ===

| Algorithm          |   Accuracy |   F1 Score |
|:-------------------|-----------:|-----------:|
| Perceptron         |     0.9551 |     0.9544 |
| Bigram (Context)   |     0.9176 |     0.9146 |
| HMM                |     0.9027 |     0.9023 |
| Unigram (Baseline) |     0.9028 |     0.8972 |


=== TABLE 2: NER RESULTS ===

| Algorithm          |   Accuracy |   F1 Score |
|:-------------------|-----------:|-----------:|
| SpaCy (CNN)        |     0.8422 |     0.8435 |
| BERT (Transformer) |     0.9616 |     0.8064 |
| CRF                |     0.9569 |     0.7393 |
